In [1]:
import kagglehub
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from skopt import gp_minimize
from skopt.space import Real, Integer, Categorical
from torch.utils.data import DataLoader
from torch.utils.data import WeightedRandomSampler
from torch.utils.data import random_split
from torch.utils.tensorboard import SummaryWriter
from torchvision import datasets, transforms



In [2]:
# Download latest version
path = kagglehub.dataset_download("subhaditya/fer2013plus")

print("Path to dataset files:", path)

100%|██████████| 132M/132M [00:05<00:00, 24.2MB/s] 

Extracting files...


Path to dataset files: /home/omarhammad/.cache/kagglehub/datasets/subhaditya/fer2013plus/versions/1


In [3]:
def create_dataloaders(train_dataset, test_dataset, batch_size, val_split=0.2):

    train_size = int((1 - val_split) * len(train_dataset))
    val_size = len(train_dataset) - train_size
    generator = torch.Generator().manual_seed(42)  # For reproducibility
    train_subset, val_subset = random_split(train_dataset, [train_size, val_size], generator=generator)

    if not hasattr(train_dataset, 'targets'):
        raise AttributeError("train_dataset must have 'targets' attribute for class balancing.")

    train_labels = [train_dataset.targets[i] for i in train_subset.indices]
    class_counts = np.bincount(train_labels)
    class_weights = 1. / np.sqrt(class_counts + 1e-6)
    class_weights = torch.tensor(class_weights, dtype=torch.float32)

    sample_weights = [class_weights[label] for label in train_labels]
    sampler = WeightedRandomSampler(sample_weights, len(sample_weights), replacement=True)

    train_loader = DataLoader(train_subset, batch_size=batch_size, sampler=sampler)
    val_loader = DataLoader(val_subset, batch_size=batch_size, shuffle=False)
    test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False)

    return train_loader, val_loader, test_loader


In [4]:
def train_model(model, train_loader, val_loader, optimizer, criterion, num_epochs=100, log_interval=100, patience=5):
    """
    Trains a given model with streamlined logging and early stopping, while logging to TensorBoard.

    Args:
        model: The model to train.
        train_loader: DataLoader for the training dataset.
        val_loader: DataLoader for the validation dataset.
        optimizer: Optimizer for training (e.g., Adam, SGD).
        criterion: Loss function (e.g., CrossEntropyLoss).
        num_epochs: Number of training epochs.
        log_interval: Frequency (in batches) to log metrics during training.
        patience: Number of epochs to wait for validation loss improvement before stopping.
    """
    # Initialize TensorBoard writer
    writer = SummaryWriter(log_dir="../logs")

    # Set device (CUDA if available, otherwise CPU)
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    model = model.to(device)

    best_val_loss = float('inf')
    patience_counter = 0
    # Training loop
    for epoch in range(num_epochs):
        model.train()
        running_loss = 0.0
        correct_train = 0
        total_train = 0

        for batch_idx, (inputs, labels) in enumerate(train_loader):
            # Move data to device
            inputs, labels = inputs.to(device), labels.to(device)

            # Zero the parameter gradients
            optimizer.zero_grad()

            # Forward pass
            outputs = model(inputs)
            loss = criterion(outputs, labels)

            # Backward pass
            loss.backward()

            # Gradient clipping
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)

            # Optimizer step
            optimizer.step()

            # Accumulate loss
            running_loss += loss.item()

            # Calculate training accuracy
            _, predicted = torch.max(outputs, 1)
            correct_train += (predicted == labels).sum().item()
            total_train += labels.size(0)

            # Log batch loss and accuracy to TensorBoard periodically
            if (batch_idx + 1) % log_interval == 0:
                avg_loss = running_loss / log_interval
                writer.add_scalar("Batch Training Loss", avg_loss, epoch * len(train_loader) + batch_idx)
                running_loss = 0.0

        # Training accuracy for the epoch
        train_accuracy = correct_train / total_train

        # Validation phase
        model.eval()
        val_loss = 0.0
        correct_val = 0
        total_val = 0
        with torch.no_grad():
            for inputs, labels in val_loader:
                inputs, labels = inputs.to(device), labels.to(device)
                outputs = model(inputs)
                loss = criterion(outputs, labels)
                val_loss += loss.item()

                # Validation accuracy
                _, predicted = torch.max(outputs, 1)
                correct_val += (predicted == labels).sum().item()
                total_val += labels.size(0)

        val_loss /= len(val_loader)
        val_accuracy = correct_val / total_val

        # Log epoch metrics to TensorBoard
        writer.add_scalar("Epoch Training Accuracy", train_accuracy, epoch)
        writer.add_scalar("Epoch Validation Accuracy", val_accuracy, epoch)
        writer.add_scalar("Epoch Validation Loss", val_loss, epoch)

        # Epoch summary
        print(f"Epoch [{epoch + 1}/{num_epochs}], Train Acc: {train_accuracy:.4f}, Val Acc: {val_accuracy:.4f}, Val Loss: {val_loss:.4f}")

        # Early stopping
        if val_loss < best_val_loss:
            best_val_loss = val_loss
            patience_counter = 0
        else:
            patience_counter += 1
            if patience_counter >= patience:
                print(f"Early stopping triggered at epoch {epoch + 1}")
                break

    writer.close()

In [5]:
def evaluate_model(model, data_loader):
    """
    Evaluates the model's performance on a given dataset.

    Args:
        model: The trained model to evaluate.
        data_loader: DataLoader for the dataset to evaluate.

    Returns:
        float: Accuracy of the model on the dataset.
    """
    # Set the device (CUDA if available, otherwise CPU)
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

    # Move model to the device
    model = model.to(device)

    model.eval()  # Set model to evaluation mode
    correct = 0
    total = 0

    with torch.no_grad():
        for inputs, labels in data_loader:
            # Move data to the same device as model
            inputs, labels = inputs.to(device), labels.to(device)

            outputs = model(inputs)
            _, predicted = torch.max(outputs, 1)
            total += labels.size(0)
            correct += (predicted == labels).sum().item()

    accuracy = correct / total
    return accuracy


In [6]:
def objective(params):
    learning_rate, batch_size, optimizer_type, dropout_rate, weight_decay, patience = params
    batch_size = int(batch_size)
    patience = int(patience)

    # Set the device (use GPU if available, otherwise CPU)
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

    # Update data loaders
    train_loader, val_loader, _ = create_dataloaders(train_dataset, test_dataset, batch_size)

    # Define model, criterion, and optimizer
    model = CNN(dropout_rate=dropout_rate).to(device)  # Pass dropout_rate to CNN
    criterion = nn.CrossEntropyLoss()

    # Use the correct optimizer with weight decay
    optimizer = getattr(optim, optimizer_type)(model.parameters(), lr=learning_rate, weight_decay=weight_decay)

    # Early stopping logic
    best_val_loss = float('inf')
    patience_counter = 0

    print(
        f"Starting training with params: lr={learning_rate}, batch_size={batch_size}, optimizer={optimizer_type}, dropout_rate={dropout_rate}, weight_decay={weight_decay}, patience={patience}, device={device}")

    for epoch in range(25):  # Use a high fixed max epoch value
        model.train()
        running_loss = 0.0

        # Training loop
        for batch_idx, (inputs, labels) in enumerate(train_loader):
            inputs, labels = inputs.to(device), labels.to(device)  # Move data to GPU
            optimizer.zero_grad()
            outputs = model(inputs)
            loss = criterion(outputs, labels)
            loss.backward()
            optimizer.step()
            running_loss += loss.item()

            # Print batch progress
            if (batch_idx + 1) % 100 == 0:  # Every 100 batches
                print(f"Epoch [{epoch + 1}/25], Batch [{batch_idx + 1}/{len(train_loader)}], Loss: {loss.item():.4f}")

        # Validation loss
        model.eval()
        val_loss = 0.0
        with torch.no_grad():
            for inputs, labels in val_loader:
                inputs, labels = inputs.to(device), labels.to(device)  # Move data to GPU
                outputs = model(inputs)
                loss = criterion(outputs, labels)
                val_loss += loss.item()
        val_loss /= len(val_loader)

        print(f"Epoch [{epoch + 1}/25], Validation Loss: {val_loss:.4f}")

        # Early stopping
        if val_loss < best_val_loss:
            print(f"Validation loss improved from {best_val_loss:.4f} to {val_loss:.4f}")
            best_val_loss = val_loss
            patience_counter = 0  # Reset patience counter if improvement
        else:
            patience_counter += 1
            print(f"No improvement in validation loss. Patience counter: {patience_counter}/{patience}")

            if patience_counter >= patience:
                print(f"Early stopping triggered after {epoch + 1} epochs")
                break

    # Calculate validation accuracy
    total, correct = 0, 0
    with torch.no_grad():
        for inputs, labels in val_loader:
            inputs, labels = inputs.to(device), labels.to(device)  # Move data to GPU
            outputs = model(inputs)
            _, predicted = torch.max(outputs, 1)
            total += labels.size(0)
            correct += (predicted == labels).sum().item()

    val_accuracy = correct / total
    print(f"Final Validation Accuracy: {val_accuracy:.4f}")
    return -val_accuracy


In [7]:
class CNN(nn.Module):
    def __init__(self, dropout_rate=0.3):
        super(CNN, self).__init__()
        self.conv1 = nn.Conv2d(1, 32, kernel_size=3, stride=1, padding=1)
        self.pool = nn.MaxPool2d(kernel_size=2, stride=2, padding=0)
        self.conv2 = nn.Conv2d(32, 64, kernel_size=3, stride=1, padding=1)
        self.conv3 = nn.Conv2d(64, 128, kernel_size=3, stride=1, padding=1)
        self.dropout = nn.Dropout(dropout_rate)
        self.fc1 = nn.Linear(128 * 6 * 6, 256)
        self.fc2 = nn.Linear(256, 8)

    def forward(self, x):
        x = self.pool(F.relu(self.conv1(x)))
        x = self.pool(F.relu(self.conv2(x)))
        x = self.pool(F.relu(self.conv3(x)))
        x = x.view(-1, 128 * 6 * 6)
        x = F.relu(self.fc1(x))
        x = self.dropout(x)
        x = self.fc2(x)
        return x


In [8]:

data_transforms = {
    "train": transforms.Compose([
        transforms.Grayscale(num_output_channels=1),
        transforms.RandomHorizontalFlip(),
        transforms.RandomRotation(10),
        transforms.RandomCrop(48, padding=4),
        transforms.ToTensor(),
        transforms.Normalize((0.5,), (0.5,))
    ]),
    "test": transforms.Compose([
        transforms.Grayscale(num_output_channels=1),
        transforms.ToTensor(),
        transforms.Normalize((0.5,), (0.5,))
    ])
}

# Paths to dataset
train_dir = path + "/fer2013plus/fer2013/train"
test_dir = path + "/fer2013plus/fer2013/test"

train_dataset = datasets.ImageFolder(train_dir, transform=data_transforms["train"])
test_dataset = datasets.ImageFolder(test_dir, transform=data_transforms["test"])


In [9]:
search_space = [
    Real(1e-6, 1e-2, name='learning_rate', prior='log-uniform'),
    Integer(16, 64, name='batch_size'),
    Categorical(['Adam', 'SGD', 'RMSprop', 'AdamW'], name='optimizer'),
    Real(0.1, 0.5, name='dropout_rate'),
    Real(1e-5, 1e-2, name='weight_decay', prior='log-uniform'),
    Integer(3, 10, name='patience')  # Early stopping patience
]


In [10]:
result = gp_minimize(
    func=objective,
    dimensions=search_space,
    n_calls=10,  # Reduced iterations
    random_state=42
)

# Define the parameter names in the same order as in the search space
param_names = ['learning_rate', 'batch_size', 'optimizer', 'dropout_rate', 'weight_decay', 'patience']

# Print each parameter with its name
print("Best parameters:")
for name, value in zip(param_names, result.x):
    print(f"{name} = {value}")

# Print the best validation accuracy
print(f"Best validation accuracy: {-result.fun:.4f}")


Starting training with params: lr=0.001535224694197351, batch_size=25, optimizer=AdamW, dropout_rate=0.33874006317859484, weight_decay=0.00021751953118777657, patience=4, device=cuda
Epoch [1/25], Batch [100/909], Loss: 1.6666
Epoch [1/25], Batch [200/909], Loss: 1.8128
Epoch [1/25], Batch [300/909], Loss: 1.7440
Epoch [1/25], Batch [400/909], Loss: 1.6306
Epoch [1/25], Batch [500/909], Loss: 1.7410
Epoch [1/25], Batch [600/909], Loss: 1.7977
Epoch [1/25], Batch [700/909], Loss: 1.7969
Epoch [1/25], Batch [800/909], Loss: 1.6868
Epoch [1/25], Batch [900/909], Loss: 1.5073
Epoch [1/25], Validation Loss: 1.4262
Validation loss improved from inf to 1.4262
Epoch [2/25], Batch [100/909], Loss: 1.5719
Epoch [2/25], Batch [200/909], Loss: 1.4680
Epoch [2/25], Batch [300/909], Loss: 1.4960
Epoch [2/25], Batch [400/909], Loss: 1.4716
Epoch [2/25], Batch [500/909], Loss: 1.7108
Epoch [2/25], Batch [600/909], Loss: 1.5335
Epoch [2/25], Batch [700/909], Loss: 1.2180
Epoch [2/25], Batch [800/909], 

In [12]:
# Extract best parameters from tuning results
best_learning_rate = result.x[0]
best_batch_size = int(result.x[1])
best_optimizer_type = result.x[2]
best_dropout_rate = result.x[3]
best_weight_decay = result.x[4]
best_patience = int(result.x[5])

# Update data loaders with the best batch size
train_loader, val_loader, test_loader = create_dataloaders(train_dataset, test_dataset, best_batch_size)

# Define the model, criterion, and optimizer
best_cnn_model = CNN(dropout_rate=best_dropout_rate)  # Pass the best dropout rate
criterion = nn.CrossEntropyLoss()

# Use the best optimizer type with weight decay
optimizer = getattr(torch.optim, best_optimizer_type)(
    best_cnn_model.parameters(),
    lr=best_learning_rate,
    weight_decay=best_weight_decay
)

# Train the model with early stopping
train_model(
    model=best_cnn_model,
    train_loader=train_loader,
    val_loader=val_loader,
    optimizer=optimizer,
    criterion=criterion,
    num_epochs=50,  # High fixed value, let early stopping decide
    patience=best_patience
)

Epoch [1/50], Train Acc: 0.3185, Val Acc: 0.4662, Val Loss: 1.5428
Epoch [2/50], Train Acc: 0.4229, Val Acc: 0.5789, Val Loss: 1.2491
Epoch [3/50], Train Acc: 0.4861, Val Acc: 0.6145, Val Loss: 1.1345
Epoch [4/50], Train Acc: 0.5181, Val Acc: 0.6168, Val Loss: 1.1264
Epoch [5/50], Train Acc: 0.5354, Val Acc: 0.6353, Val Loss: 1.0562
Epoch [6/50], Train Acc: 0.5589, Val Acc: 0.6552, Val Loss: 1.0098
Epoch [7/50], Train Acc: 0.5723, Val Acc: 0.6638, Val Loss: 0.9578
Epoch [8/50], Train Acc: 0.5872, Val Acc: 0.6703, Val Loss: 0.9493
Epoch [9/50], Train Acc: 0.5969, Val Acc: 0.6832, Val Loss: 0.9014
Epoch [10/50], Train Acc: 0.6073, Val Acc: 0.6826, Val Loss: 0.9113
Epoch [11/50], Train Acc: 0.6087, Val Acc: 0.6847, Val Loss: 0.8807
Epoch [12/50], Train Acc: 0.6198, Val Acc: 0.6969, Val Loss: 0.8687
Epoch [13/50], Train Acc: 0.6248, Val Acc: 0.7017, Val Loss: 0.8445
Epoch [14/50], Train Acc: 0.6340, Val Acc: 0.6978, Val Loss: 0.8595
Epoch [15/50], Train Acc: 0.6358, Val Acc: 0.7089, Val Lo

In [13]:
train_accuracy = evaluate_model(best_cnn_model, train_loader)
print(f"Train Accuracy: {train_accuracy:.4f}")

Train Accuracy: 0.7069


In [14]:
test_accuracy = evaluate_model(best_cnn_model, test_loader)
print(f"Test Accuracy: {test_accuracy:.4f}")

Test Accuracy: 0.7364


In [15]:
# Save the model's state_dict
torch.save(best_cnn_model.state_dict(), 'best_cnn_model.pth')